In [26]:
import numpy as np
import yaml

In [27]:
# open input.yaml
with open("input.yaml", "r") as f:
    input_file = yaml.safe_load(f)

# automatic generation of many displacement-controlled steps
n_steps = input_file["n_steps"]
print(f"n_steps from input.yaml = {n_steps}")

n_steps = input_file["n_steps"]
u_max = 0.001 # 1 mm

s1 = int(n_steps * 0.3)
s2 = int(n_steps * 0.5)
s3 = n_steps - s1 - s2

u_vals = np.concatenate([
    np.linspace(0, 0.0004, int(s1)),
    np.linspace(0.00041, 0.0006, int(s2)),
    np.linspace(0.00061, u_max, int(n_steps-s1-s2))
])

steps = [IndentedList([float(u), 0.0]) for u in u_vals]

print(f"steps generated = {len(steps)}")

n_steps from input.yaml = 20
steps generated = 20


In [28]:
import yaml

class IndentedList(list): pass
def indented_list_presenter(dumper, data):
    return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)

yaml.add_representer(IndentedList, indented_list_presenter)

def generate_z3st_bc(filename="boundary_conditions.yaml"):
    # Structure
    bc_data = {
        "mechanical": {
            "steel": [
                {"type": "Clamp_x", "region": "xmax"},
                {"type": "Clamp_y", "region": "ymin"},
                {
                    "region": "ymax",
                    "type": "Dirichlet",
                    "displacement": steps
                }
            ]
        },
        "thermal": {
            "steel": [
                {
                    "type": "Dirichlet", 
                    "region": "xmax", 
                    "temperature": 300.0
                }
            ]
        },
        "damage": {
            "steel": [
                {
                    "type": "Dirichlet", 
                    "region": "crack", 
                    "value": 1.0
                }
            ]
        }
    }

    # Writing
    with open(filename, 'w') as file:
        yaml.dump(bc_data, file, sort_keys=False, default_flow_style=False, indent=2)
    
    print(f"'{filename}' generated.")

if __name__ == "__main__":
    generate_z3st_bc()

'boundary_conditions.yaml' generated.
